<a href="https://colab.research.google.com/github/kanishka1114/fairness-auditor-ai/blob/main/AUDITING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Required Libraries

In [1]:
pip install pandas scikit-learn fairlearn matplotlib

Loading Dataset

In [2]:
import pandas as pd
from sklearn.datasets import fetch_openml

data = fetch_openml(name='adult', version=2, as_frame=True)

df = data.frame

print(df.head())

   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country  class  
0       

Data Preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = df.dropna()

label = LabelEncoder()

for col in df.columns:
    # Modified condition to include 'category' dtype
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        df[col] = label.fit_transform(df[col])

X = df.drop("class", axis=1)
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

/tmp/ipykernel_13879/2478268472.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = label.fit_transform(df[col])
/tmp/ipykernel_13879/2478268472.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = label.fit_transform(df[col])
/tmp/ipykernel_13879/2478268472.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

Training basic ML model ( LOGISTIC REGRESSION )

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit (X_train, y_train)

Evaluating Normal Performance

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Fairness Auditing

In [ ]:
sensitive_feature = X_test["sex"]

Fairness Metrics

In [ ]:
from fairlearn.metrics import demographic_parity_difference
from fairlearn.metrics import equalized_odds_difference

dp = demographic_parity_difference(
    y_test,
    y_pred,
    sensitive_features=sensitive_feature
)

eo = equalized_odds_difference(
    y_test,
    y_pred,
    sensitive_features=sensitive_feature
)

print("Demographic Parity Difference:", dp)
print("Equalized Odds Difference:", eo)

Visualize Fairness

In [ ]:
from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score

metrics = MetricFrame(
    metrics=accuracy_score,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_feature
)

print(metrics.by_group)

Implementing Bias Mitigation

In [ ]:
from fairlearn.reductions import ExponentiatedGradient
from fairlearn.reductions import DemographicParity

mitigator = ExponentiatedGradient(
    LogisticRegression(),
    constraints=DemographicParity()
)

mitigator.fit(X_train, y_train, sensitive_features=X_train["sex"])

y_pred_mitigated = mitigator.predict(X_test)

Evaluating after mitigation

In [ ]:
dp_after = demographic_parity_difference(
    y_test,
    y_pred_mitigated,
    sensitive_features=sensitive_feature
)

print("DP after mitigation:", dp_after)

Comparing performance (Small loss but more fair model.)

In [ ]:
print("Original Accuracy:", accuracy_score(y_test, y_pred))
print("Fair Model Accuracy:", accuracy_score(y_test, y_pred_mitigated))

Train Model
 ↓
Check fairness
 ↓
If bias > threshold
 ↓
Apply mitigation
 ↓
Generate report

Automated System Code

In [ ]:
def fairness_pipeline(model, X_train, X_test, y_train, y_test, sensitive_feature):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    dp = demographic_parity_difference(
        y_test,
        y_pred,
        sensitive_features=sensitive_feature
    )

    print("Initial DP:", dp)

    if abs(dp) > 0.1:

        print("Bias detected → applying mitigation")

        mitigator = ExponentiatedGradient(
            LogisticRegression(),
            constraints=DemographicParity()
        )

        mitigator.fit(X_train, y_train,
                      sensitive_features=X_train["sex"])

        y_pred = mitigator.predict(X_test)

    print("Final Accuracy:", accuracy_score(y_test, y_pred))

    return y_pred

pipeline

In [ ]:
fairness_pipeline(
    LogisticRegression(),
    X_train,
    X_test,
    y_train,
    y_test,
    sensitive_feature
)

In [ ]:
print("Fairness Auditor Running Successfully")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)